In [2]:
from pathlib import Path

from satchange.data import prepare_levir_cd
from satchange.optimization import run_parameter_search
from satchange.utils import load_config

current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

config_path = project_root / "configs" / "siam_mit_b3_diff_resized.yaml"
config = load_config(config_path)

print("Config:", config_path)
print("Model:", config["model"]["name"])
print("Image size:", config["data"]["image_size"])
print("Precision:", config["runtime"]["precision"])

Config: configs/siam_mit_b3_diff_resized.yaml
Model: siam_mit_b3_diff
Image size: 256
Precision: bf16


In [3]:
RUN_DATA_SETUP = False  # LEVIR-CD уже подготовлен.

dataset_root = project_root / config["dataset"]["prepared_directory"]
if RUN_DATA_SETUP:
    dataset_root = prepare_levir_cd(config=config, project_root=project_root)

THRESHOLDS = [round(value / 100, 2) for value in range(5, 96, 5)]
loss_config = config["training"]["loss"]

print("Dataset:", dataset_root)
print("Thresholds:", THRESHOLDS)

Dataset: data/LEVIR-CD
Thresholds: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]


In [5]:
HPO_EPOCHS = 40
HPO_GPU_IDS = [0, 1, 2, 3]  # Только выделенные тебе свободные GPU.
HPO_OUTPUT_ROOT = project_root / "artifacts" / "hyperparameter_search"

FIXED_PARAMETERS = {
    "scheduler": config["training"]["scheduler"],
    "positive_weight": loss_config["pos_weight"],
    "bce_weight": loss_config["bce_weight"],
    "dice_weight": loss_config["dice_weight"],
    "batch_size": config["data"]["batch_size"],
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "seed": config["project"]["seed"],
    "precision": config["runtime"]["precision"],
    "model_parameters": config["model"]["parameters"],
}


# Эти параметры теперь перебираются, а не фиксируются.
FIXED_PARAMETERS.pop("batch_size", None)
FIXED_PARAMETERS.pop("learning_rate", None)

# Пока оставляем исходный weight decay.
FIXED_PARAMETERS["weight_decay"] = 1e-2
SEARCH_NAME = (
    f"heavy_resized_stage1_batch4_8_12_16_lr_"
    f"{HPO_EPOCHS}ep_seed{FIXED_PARAMETERS['seed']}_"
    f"{FIXED_PARAMETERS['precision']}_v1"
)

SEARCH_SPACE = {
    "batch_size": [4, 8, 12, 16],
    "learning_rate": [3e-5, 1e-4, 3e-4],
}




In [6]:
stage1_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=HPO_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=10,
)

print("Results:", HPO_OUTPUT_ROOT / SEARCH_NAME)

Trials: 12
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=3e-05 | wd=0.01 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=256 | in_channels=3 | out_channels=1 | encoder_depths=[3, 4, 18, 3] | decoder_channels=128 | refinement_channels=[64, 32] | norm_groups=8 | dropout=0.0 | attention_dropout=0.0 | drop_path_rate=0.1 | pretrained_backbone=False | use_checkpointing=False | scheduler=none
epochs=40

[START] Trial 001 -> GPU 1
seed=42 | lr=0.0001 | wd=0.01 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=256 | in_channels=3 | out_channels=1 | encoder_depths=[3, 4, 18, 3] | decoder_channels=128 | refinement_channels=[64, 32] | norm_groups=8 | dropout=0.0 | attention_dropout=0.0 | drop_path_rate=0.1 | pretrained_backbone=False | use_checkpointing=False | scheduler=none
epochs=40

[START] Trial 002 -> GPU 2
seed=42 | lr=0.0003 | wd=0.01 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=256 | in_channels=3 | out_channels=1 | encoder_depths=[3, 4, 18, 3] | decoder_channels=128 

поставить для resized другие парамметры. ПРо тупарылый не понимает контекста